# POC — Criação e Carga das Tabelas Dimensão
### Origem: `tb_dados_bruto` → Destino: tabelas dimensão
---
| # | Tabela destino   | Colunas chave origem                          |
|---|------------------|-----------------------------------------------|
| 1 | `tb_contato`     | `id_dsc_contato`, `nm_dsc_contato`            |
| 2 | `tb_lista`       | `id_dsc_lista`, `nm_dsc_lista`                |
| 3 | `tb_mailing`     | `id_dsc_mailing`, `nm_dsc_mailing`            |
| 4 | `tb_status`      | `id_dsc_status`, `nm_dsc_status`, grupos      |
| 5 | `tb_tabulacao`   | `id_dsc_tabulacao`, `nm_dsc_tabulacao`        |
| 6 | `tb_usuario`     | `id_dsc_usuario`, `nm_dsc_usuario`            |

## 1. Imports

In [1]:
from sqlalchemy import create_engine, text
import time

## 2. Configurações de conexão


In [2]:
# ── Configurações ──────────────────────────────────────────────
DB_HOST     = 'localhost'
DB_PORT     = 3306
DB_USER     = 'root'       # altere para seu usuário
DB_PASSWORD = '123456'  # altere para sua senha
DB_NAME     = 'db_poc_call_center'     # altere se necessário
# ───────────────────────────────────────────────────────────────

connection_string = (
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)

engine = create_engine(connection_string, echo=False)

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'Conexão OK!' AS status"))
    print(result.fetchone()[0])

Conexão OK!


## 3. DDL — Criação das tabelas dimensão

- `num_index` → `INT AUTO_INCREMENT PRIMARY KEY`
- Colunas `id_*` → `INT` (dados convertidos via `CAST` no INSERT)
- Colunas `nm_*` / `tp_*` → `VARCHAR(100)`
- `UNIQUE KEY` nas colunas `id_*` para evitar duplicatas na carga

In [3]:
DDL_TABLES = [

    # ── 1. tb_contato ────────────────────────────────────────────
    """
    CREATE TABLE IF NOT EXISTS tb_contato (
        num_index      INT          NOT NULL AUTO_INCREMENT,
        id_dsc_contato INT          NOT NULL,
        nm_dsc_contato VARCHAR(100) NULL,

        PRIMARY KEY (num_index)

    ) ENGINE=InnoDB
      DEFAULT CHARSET=utf8mb4
      COLLATE=utf8mb4_unicode_ci
      COMMENT='Dimensão Contato';    
    """,

    # ── 2. tb_lista ──────────────────────────────────────────────
    """
    CREATE TABLE IF NOT EXISTS tb_lista (
        num_index    INT          NOT NULL AUTO_INCREMENT,
        id_dsc_lista INT          NOT NULL,
        nm_dsc_lista VARCHAR(100) NULL,

        PRIMARY KEY (num_index)

    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
      COMMENT='Dimensão Lista'
    """,

    # # ── 3. tb_mailing ────────────────────────────────────────────
    """
    CREATE TABLE IF NOT EXISTS tb_mailing (
        num_index      INT          NOT NULL AUTO_INCREMENT,
        id_dsc_mailing INT          NOT NULL,
        nm_dsc_mailing VARCHAR(100) NULL,

        PRIMARY KEY (num_index)

    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
      COMMENT='Dimensão Mailing'
    """,

    # # ── 4. tb_status ─────────────────────────────────────────────
    """
    CREATE TABLE IF NOT EXISTS tb_status (
        num_index     INT          NOT NULL AUTO_INCREMENT,
        id_dsc_status INT          NOT NULL,
        nm_dsc_status VARCHAR(100) NULL,
        nm_dsc_grupo  VARCHAR(100) NULL,
        nm_sub_grupo  VARCHAR(100) NULL,

        PRIMARY KEY (num_index)

    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
      COMMENT='Dimensão Status'
    """,

    # # ── 5. tb_tabulacao ──────────────────────────────────────────
    """
    CREATE TABLE IF NOT EXISTS tb_tabulacao (
        num_index        INT          NOT NULL AUTO_INCREMENT,
        id_dsc_tabulacao INT          NOT NULL,
        nm_dsc_tabulacao VARCHAR(100) NULL,

        PRIMARY KEY (num_index)
        
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
      COMMENT='Dimensão Tabulação'
    """,

    # # ── 6. tb_usuario ────────────────────────────────────────────
    """
    CREATE TABLE IF NOT EXISTS tb_usuario (
        num_index      INT          NOT NULL AUTO_INCREMENT,
        id_dsc_usuario INT          NOT NULL,
        nm_dsc_usuario VARCHAR(100) NULL,

        PRIMARY KEY (num_index)

    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
      COMMENT='Dimensão Usuário'
    """,
]

with engine.connect() as conn:
    for ddl in DDL_TABLES:
        conn.execute(text(ddl))
    conn.commit()

print("✅ Todas as tabelas dimensão foram criadas (ou já existiam)!")

✅ Todas as tabelas dimensão foram criadas (ou já existiam)!


## 4. DML — Carga das tabelas dimensão

Cada `INSERT ... SELECT` usa `CAST(coluna AS UNSIGNED)` para converter  
as colunas `id_*` de `VARCHAR` → `INT` durante a extração da `tb_dados_bruto`.  

`INSERT IGNORE` garante **idempotência**: re-executar o notebook não duplica registros  
(ignora conflitos na `UNIQUE KEY`).

> ℹ️ Valores vazios `''` em colunas `id_*` retornam `0` no `CAST` — o `WHERE` filtra esses casos.

In [6]:
DML_INSERTS = {

    # ── 1. tb_contato ────────────────────────────────────────────
    'tb_contato': """
        INSERT IGNORE INTO tb_contato
            (id_dsc_contato, nm_dsc_contato)
        SELECT
            CAST(id_dsc_contato AS UNSIGNED),
            tp_dsc_contato
        FROM tb_dados_bruto
        WHERE id_dsc_contato IS NOT NULL
          AND id_dsc_contato <> ''
        GROUP BY id_dsc_contato, nm_dsc_status, tp_dsc_contato
    """,

    # ── 2. tb_lista ──────────────────────────────────────────────
    'tb_lista': """
        INSERT IGNORE INTO tb_lista
            (id_dsc_lista, nm_dsc_lista)
        SELECT
            CAST(id_dsc_lista AS UNSIGNED),
            nm_dsc_lista
        FROM tb_dados_bruto
        WHERE id_dsc_lista IS NOT NULL
          AND id_dsc_lista <> ''
        GROUP BY id_dsc_lista, nm_dsc_lista
    """,

    # ── 3. tb_mailing ────────────────────────────────────────────
    'tb_mailing': """
        INSERT IGNORE INTO tb_mailing
            (id_dsc_mailing, nm_dsc_mailing)
        SELECT
            CAST(id_dsc_mailing AS UNSIGNED),
            nm_dsc_mailing
        FROM tb_dados_bruto
        WHERE id_dsc_mailing IS NOT NULL
          AND id_dsc_mailing <> ''
        GROUP BY id_dsc_mailing, nm_dsc_mailing
    """,

    # ── 4. tb_status ─────────────────────────────────────────────
    'tb_status': """
        INSERT IGNORE INTO tb_status
            (id_dsc_status, nm_dsc_status, nm_dsc_grupo, nm_sub_grupo)
        SELECT
            CAST(id_dsc_status AS UNSIGNED),
            nm_dsc_status,
            nm_dsc_grupo,
            nm_sub_grupo
        FROM tb_dados_bruto
        WHERE id_dsc_status IS NOT NULL
          AND id_dsc_status <> ''
        GROUP BY id_dsc_status, nm_dsc_status, nm_dsc_grupo, nm_sub_grupo
    """,

    # ── 5. tb_tabulacao ──────────────────────────────────────────
    'tb_tabulacao': """
        INSERT IGNORE INTO tb_tabulacao
            (id_dsc_tabulacao, nm_dsc_tabulacao)
        SELECT
            CAST(id_dsc_tabulacao AS UNSIGNED),
            nm_dsc_tabulacao
        FROM tb_dados_bruto
        WHERE id_dsc_tabulacao IS NOT NULL
          AND id_dsc_tabulacao <> ''
        GROUP BY id_dsc_tabulacao, nm_dsc_tabulacao
    """,

    # ── 6. tb_usuario ────────────────────────────────────────────
    'tb_usuario': """
        INSERT IGNORE INTO tb_usuario
            (id_dsc_usuario, nm_dsc_usuario)
        SELECT
            CAST(id_dsc_usuario AS UNSIGNED),
            nm_dsc_usuario
        FROM tb_dados_bruto
        WHERE id_dsc_usuario IS NOT NULL
          AND id_dsc_usuario <> ''
        GROUP BY id_dsc_usuario, nm_dsc_usuario
    """,
}

print("🚀 Iniciando carga das dimensões...\n")
start_time = time.time()

with engine.connect() as conn:
    for table, dml in DML_INSERTS.items():
        result = conn.execute(text(dml))
        conn.commit()
        print(f"   ✅ {table:<18} — {result.rowcount:>6,} linha(s) inserida(s)")

elapsed = time.time() - start_time
print(f"\n✅ Carga concluída em {elapsed:.2f}s")

🚀 Iniciando carga das dimensões...

   ✅ tb_contato         —     14 linha(s) inserida(s)
   ✅ tb_lista           —      6 linha(s) inserida(s)
   ✅ tb_mailing         —     16 linha(s) inserida(s)
   ✅ tb_status          —     21 linha(s) inserida(s)
   ✅ tb_tabulacao       —     28 linha(s) inserida(s)
   ✅ tb_usuario         —      7 linha(s) inserida(s)

✅ Carga concluída em 14.01s


## 5. Validação — contagem por tabela

In [ ]:
TABELAS = list(DML_INSERTS.keys())

print(f"{'Tabela':<20} {'Registros':>10}")
print("-" * 32)

with engine.connect() as conn:
    for tabela in TABELAS:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {tabela}")).scalar()
        print(f"{tabela:<20} {count:>10,}")

## 6. Preview por tabela

In [ ]:
import pandas as pd

for tabela in TABELAS:
    print(f"\n{'─'*55}")
    print(f"  📋 {tabela}")
    print(f"{'─'*55}")
    df_preview = pd.read_sql(f"SELECT * FROM {tabela} LIMIT 5", con=engine)
    display(df_preview)